## Load Libraries

In [1]:
import os # For interacting with the operating system
import requests # For making HTTP requests
import subprocess # For running external commands
from dotenv import load_dotenv # For loading environment variables
from IPython.display import Markdown, display  # For displaying formatted Markdown in Jupyter notebooks
from openai import OpenAI # For interacting with the OpenAI API
from pathlib import Path # For path and location
import glob
import re

## Load Environment Key

In [2]:
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

# Go up one level to the project folder
project_dir = os.path.dirname(script_dir)
env_path = os.path.join(project_dir, "env_keys", ".env")

# Access the variable
load_dotenv(dotenv_path=env_path)
openai_api_key = os.getenv("OPENAI_API_KEY")
print("API Key loaded:", openai_api_key is not None)

API Key loaded: True


## Ollma Initialize

In [3]:
subprocess.Popen("ollama serve", shell=True)

<Popen: returncode: None args: 'ollama serve'>

In [4]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [5]:
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

NAME                ID              SIZE      MODIFIED    
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    3 weeks ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    3 weeks ago    



In [6]:
OLLAMA_API_URL = "http://localhost:11434/v1"
OPENAI_API_URL = "https://api.openai.com/v1"

## Initialize OpenAI Client for Ollma

In [7]:
openai = OpenAI(base_url=OPENAI_API_URL, api_key=openai_api_key)
ollama = OpenAI(base_url=OLLAMA_API_URL, api_key="None")

## Prompt

In [8]:
tell_a_joke = [
    {
        "role": "user",
        "content" :  "Tell a joke for student who are new to Calculus"
    }
]

In [9]:
ollama_response = ollama.chat.completions.create(model="llama3.2", temperature=0.2, messages=tell_a_joke)
display(Markdown(ollama_response.choices[0].message.content))

Here's one:

Why did the derivative go to therapy?

Because it was feeling a little "unstable"!

Get it? In calculus, we talk about derivatives and stability, but in this joke, the derivative is literally unstable because it's going to therapy! It's a play on words that might make you laugh (or groan) as a new student of calculus.

Or how about another one:

Why did the integral go to the party?

Because it was a "sum"-mer event!

This one plays on the idea of integrals being sums, but also references the summer season. I hope it brings a smile to your face!

In [10]:
# openai_response = openai.chat.completions.create(model="gpt-4.1-nano", temperature=0.2, messages=tell_a_joke)
# display(Markdown(openai_response.choices[0].message.content))

## Load Employee Data

In [11]:
knowledge = {}

In [12]:
files = glob.glob("knowledge-base/employees/*")

In [13]:
for file in files:
    name = Path(file).stem.split(" ")[-1]
    with open(file, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [14]:
Markdown(knowledge['walker'][:210])

# HR Record

# Brandon Walker

## Summary
- **Date of Birth:** December 5, 1993
- **Job Title:** Technical Support Specialist
- **Location:** Remote (Based in Phoenix, Arizona)
- **Current Salary:** $62,000

##

## Load Prodcut Data

In [15]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [16]:
Markdown(knowledge['markellm'][:210])

# Product Summary

# Markellm

## Summary

Markellm is an innovative two-sided marketplace designed to seamlessly connect consumers with insurance companies. Powered by advanced matching AI, Markellm transforms

In [53]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [45]:
message = "Who is markellm ?"

In [35]:
def get_relevant_context(message):
    words = (re.findall(r"[A-Za-z]+", message))
    relevant_knowledge = [knowledge[word.lower()] for word in words if word in knowledge]
    return relevant_knowledge

In [47]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if relevant_context:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    else:
        result = "There is no additional context relevant to the user's question."
    return result

In [ ]:
def chat(message):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="llama3.2", temperature=0.2, messages=messages)
    return response.choices[0].message.content

In [60]:
Markdown(chat(message="Who is walker ?"))

Walker refers to Brandon Walker, a Technical Support Specialist at Insurellm, based in Phoenix, Arizona.